GDELT → ADF → ADLS → Auto Loader → Bronze Delta ✅

In [0]:
%sql
select * from gdelt_dev.bronze.gdelt_gkg
limit 5;

In [0]:
from pyspark.sql import functions as F

bronze_df = spark.table("gdelt_dev.bronze.gdelt_gkg")

column_check = (
    bronze_df.withColumn(
        "field_count",
        F.size(F.split(F.col("raw_record"), "/t")
    )
)
)

In [0]:
display(column_check.groupby("field_count").count().orderBy(F.desc("count")))

In [0]:
rows = (
    bronze_df
    .select("raw_record")
    .limit(10)
    .collect()
)

for i, row in enumerate(rows):
    print(f"\n--- ROW {i+1} ---")
    print(repr(row["raw_record"][:1000]))

In [0]:
from pyspark.sql import functions as F

debug_df = (
    bronze_df
    .withColumn(
        "field_count",
        F.size(F.split(F.col("raw_record"), "\t"))
    )
    .withColumn(
        "record_length",
        F.length("raw_record")
    )
    .withColumn(
        "starts_with_gkg_id",
        F.col("raw_record").rlike(r"^\d{14}-(?:\d+|T\d+)\t")
    )
)

display(
    debug_df
    .groupBy(
        "field_count",
        "starts_with_gkg_id"
    )
    .count()
    .orderBy(
        "field_count",
        "starts_with_gkg_id"
    )
)

In [0]:
display(
    debug_df
    .filter(F.col("field_count") == 1)
    .select(
        "source_file_name",
        "record_length",
        "starts_with_gkg_id",
        "raw_record"
    )
    .limit(10)
)

In [0]:
gkg_columns = [
    "GKGRECORDID",
    "DATE",
    "SourceCollectionIdentifier",
    "SourceCommonName",
    "DocumentIdentifier",
    "Counts",
    "V2Counts",
    "Themes",
    "V2Themes",
    "Locations",
    "V2Locations",
    "Persons",
    "V2Persons",
    "Organizations",
    "V2Organizations",
    "V2Tone",
    "Dates",
    "GCAM",
    "SharingImage",
    "RelatedImageEmbeds",
    "SocialImageEmbeds",
    "SocialVideoEmbeds",
    "Quotations",
    "AllNames",
    "Amounts",
    "TranslationInfo",
    "Extras"
]

print(len(gkg_columns))

In [0]:
from pyspark.sql import functions as F


def parse_gkg(df):

    fields = F.split(
        F.col("raw_record"),
        "\t"
    )

    parsed = df.withColumn(
        "_fields",
        fields
    )

    parsed = parsed.withColumn(
        "_field_count",
        F.size(F.col("_fields"))
    )

    for index, column_name in enumerate(gkg_columns):
        parsed = parsed.withColumn(
            column_name,
            F.col("_fields").getItem(index)
        )

    return parsed

In [0]:
parsed_text = parse_gkg(bronze_df)

display(
    parsed_text.select(
        "GKGRECORDID",
        "DATE",
        "SourceCommonName",
        "DocumentIdentifier",
        "V2Themes",
        "V2Organizations",
        "V2Locations",
        "V2Tone",
        "Extras",
        "_field_count"
    ).limit(10)
)

In [0]:
%sql
select count(*) from gdelt_dev.bronze.gdelt_gkg

Extract the article title

In [0]:
display(
    parsed_text.select(
        "Extras",
        F.regexp_extract(
            F.col("Extras"),
            r"(?s)<PAGE_TITLE>(.*?)</PAGE_TITLE>",
            1
        ).alias("extracted_title")
    ).limit(20)
)

In [0]:
display(parsed_text)

In [0]:
parsed_text.printSchema()

In [0]:
parsed_text = parsed_text.withColumn(
        "extracted_title",
        F.regexp_extract(
            F.col("Extras"),
            r"(?s)<PAGE_TITLE>(.*?)</PAGE_TITLE>",
            1
        ).alias("extracted_title")
    )

In [0]:
parsed_text.printSchema()

In [0]:
import html
from pyspark.sql.types import StringType

@F.udf(returnType=StringType())
def html_unescape(text):
    if text is None:
        return None
    
    return html.unescape(text)

In [0]:
parsed_text = parsed_text.withColumn(
    "title",
    html_unescape(F.col("extracted_title"))
)

In [0]:
display(parsed_text.select("title", "extracted_title"))

# **Convert Date into a proper timestampe**

In [0]:
parsed_text = parsed_text.withColumn(
    "published_at",
    F.to_timestamp(
        F.col("DATE"),
        "yyyyMMddHHmmss"
)
)

display(parsed_text.select("DATE", "published_at"))
    


In [0]:
parsed_text = parsed_text.withColumn(
    "article_url",
    F.col("DocumentIdentifier")
)

In [0]:
display(parsed_text.select("DocumentIdentifier", "article_url").limit(20))

In [0]:
parsed_text = parsed_text.withColumn(
    "source_domain",
    F.parse_url(
        F.col("article_url"),
        F.lit("HOST")
    )
)

In [0]:
display(parsed_text.select("source_domain", "article_url").limit(20))

In [0]:
parsed_text = parsed_text.withColumn(
    "tone",
    F.split(
        F.col("V2Tone"),
        ","
    ).getItem(0).cast("double")
)

In [0]:
display(parsed_text.select("title", "tone").limit(5))

In [0]:
parsed_text = parsed_text.withColumn(
    "themes",
    F.split(
        F.col("V2Themes"),
        ";"
    )
)

display(parsed_text.select("Themes").limit(20))

In [0]:
display(
    parsed_text
    .select(
        "title",
        F.explode("themes").alias("theme")
    )
    .groupBy("theme")
    .count()
    .orderBy(F.desc("count"))
    .limit(30)
)

In [0]:
print(type(parsed_text))

In [0]:
display(parsed_text.select("V2Organizations").limit(20))

In [0]:
parsed_text = parsed_text.withColumn(
    "organizations_raw",
    F.split(
        F.col("V2Organizations"),
        ";"
    )
).withColumn(
    "persons_raw",
    F.split(F.col("V2Persons"), ";")
)

display(parsed_text.select("organizations_raw","persons_raw").limit(10))

In [0]:
parsed_text.printSchema()

# Create a clean transformation function

In [0]:
def transform_gkg_to_articles(df):

    parsed = parse_gkg(df)

    valid = parsed.filter(
        F.col("_field_count") == 27
    )

    articles = (
        valid

        .withColumn(
            "title_raw",
            F.regexp_extract(
                F.col("Extras"),
                r"<PAGE_TITLE>(.*?)</PAGE_TITLE>",
                1
            )
        )

        .withColumn(
            "title",
            html_unescape(F.col("title_raw"))
        )

        .withColumn(
            "published_at",
            F.to_timestamp(
                F.col("DATE"),
                "yyyyMMddHHmmss"
            )
        )

        .withColumn(
            "article_url",
            F.col("DocumentIdentifier")
        )

        .withColumn(
            "source_domain",
            F.parse_url(
                F.col("DocumentIdentifier"),
                F.lit("HOST")
            )
        )

        .withColumn(
            "tone",
            F.split(
                F.col("V2Tone"),
                ","
            ).getItem(0).cast("double")
        )

        .withColumn(
            "themes",
            F.split(
                F.col("V2Themes"),
                ";"
            )
        )

        .withColumn(
            "organizations_raw",
            F.split(
                F.col("V2Organizations"),
                ";"
            )
        )

        .withColumn(
            "persons_raw",
            F.split(
                F.col("V2Persons"),
                ";"
            )
        )
    )

    return articles

# Select the actual Silver schema

In [0]:
def select_silver_columns(df):

    return df.select(

        F.col("GKGRECORDID")
            .alias("article_id"),

        "published_at",

        "article_url",

        "source_domain",

        F.col("SourceCommonName")
            .alias("source_name"),

        "title",

        "title_raw",

        "tone",

        "themes",

        "organizations_raw",

        "persons_raw",

        F.col("V2Locations")
            .alias("locations_raw"),

        F.col("TranslationInfo")
            .alias("translation_info"),

        F.col("Extras")
            .alias("extras_raw"),

        "source_file_name",

        "source_file_path",

        "ingestion_date",

        F.col("ingested_at")
            .alias("bronze_ingested_at"),

        F.current_timestamp()
            .alias("silver_processed_at")
    )

In [0]:
test_articles = (
    bronze_df
    .transform(transform_gkg_to_articles)
    .transform(select_silver_columns)
)

display(test_articles.limit(10))

In [0]:
test_articles.filter(
    F.col("article_id").isNull()
).count()

In [0]:
test_articles.select(
    F.count("*").alias("total"),
    F.sum(
        F.when(
            F.col("title").isNull() |
            (F.trim(F.col("title")) == ""),
            1
        ).otherwise(0)
    ).alias("missing_titles")
).show()

In [0]:
display(
    test_articles.
    groupBy("article_id")
    .count()
    .filter("count > 1")
)

In [0]:
silver_checkpoint_path = (
    "/Volumes/gdelt_dev/raw/gdelt_control/"
    "checkpoints/silver_articles/"
)

In [0]:
bronze_stream = (
    spark.readStream
    .table(
        "gdelt_dev.bronze.gdelt_gkg"
    )
)

In [0]:
silver_stream = (
    bronze_stream
    .transform(transform_gkg_to_articles)
    .transform(select_silver_columns)
)

In [0]:
silver_query = (
    silver_stream
    .writeStream
    .format("delta")
    .option(
        "checkpointLocation",
        silver_checkpoint_path
    )
    .trigger(
        availableNow=True
    )
    .toTable(
        "gdelt_dev.silver.articles"
    )   
)

In [0]:
%sql
select
    article_id,
    published_at,
    article_url,
    source_domain,
    title,
    tone,
    themes
from gdelt_dev.silver.articles
where title is not null
and trim(title) <> ''
order by published_at desc
limit 20;

In [0]:
%sql
select count(*)
from gdelt_dev.bronze.gdelt_gkg

In [0]:
%sql
select count(*) from gdelt_dev.silver.articles

In [0]:
%sql
SELECT
    theme,
    COUNT(*) AS article_count
FROM (
    SELECT
        explode(themes) AS theme
    FROM gdelt_dev.silver.articles
)
WHERE theme IS NOT NULL
  AND theme <> ''
GROUP BY theme
ORDER BY article_count DESC;

In [0]:
%sql
SELECT
    source_file_name,
    COUNT(*) AS articles
FROM gdelt_dev.silver.articles
GROUP BY source_file_name
ORDER BY source_file_name DESC;